# 02 — Prime Gaps

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** measure how the ordered prime sequence continues across scale through gap structure.

Prime gaps are defined by

\[
g_n = p_{n+1} - p_n
\]

Notebook 01 showed a discrete residue constraint.  
Notebook 02 shifts to ordered structure:

> Residues show what remains under constraint.  
> Gaps show how the prime sequence continues.  
> Drift measures deviation from expected logarithmic scaling.

## 0. Setup

This notebook uses the flat artifact structure:

```text
02_prime_gaps/
├── data/
├── docs/
├── figures/
└── tex/
```

The export zip is written at repo root as:

```text
02_prime_gaps_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "02_prime_gaps"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Prime Gaps"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Prime gaps measure local spacing in the ordered prime sequence.

This notebook uses three terms consistently:

- **continues:** prime order progresses across scale.
- **remains under constraint / persists:** measured gap structure stays near a stated baseline.
- **drift:** measurable deviation from expected scaling.

For gaps, the first scaling baseline is logarithmic:

\[
\mathbb{E}[g_n] \approx \log(p_n)
\]

This is not a prediction of each individual gap. It is a coarse density-scale baseline.

## 2. Constraint definition

Let \(p_n\) denote the \(n\)-th prime.

The prime gap is

\[
g_n = p_{n+1} - p_n
\]

The baseline scale near \(p_n\) is

\[
g_n \sim \log(p_n)
\]

So a simple drift measurement is

\[
drift_n = \frac{|g_n - \log(p_n)|}{\log(p_n)}
\]

This notebook does not claim gaps are random or fully predictable.  
It measures structure and deviation.

In [ ]:
# Parameters

N_MAX = 1_000_000
RANDOM_SEED = 9423

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
    "constraint": "prime gaps compared to logarithmic scaling",
}

params

## 3. Data generation

Generate primes below \(N_{\max}\), then compute gaps.

This uses a local sieve so the notebook has no required external dependency.

In [ ]:
def sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n:i] = False
    return np.nonzero(s)[0]

primes = sieve(N_MAX)

gap_start_primes = primes[:-1]
gap_end_primes = primes[1:]
gaps = np.diff(primes)
log_baseline = np.log(gap_start_primes)

summary = {
    "n_max": int(N_MAX),
    "prime_count": int(len(primes)),
    "gap_count": int(len(gaps)),
    "first_primes": primes[:10].tolist(),
    "first_gaps": gaps[:10].tolist(),
    "last_primes": primes[-10:].tolist(),
    "last_gaps": gaps[-10:].tolist(),
}

summary

## 4. Measurement

Compute basic gap statistics, logarithmic baseline error, and drift.

Because early primes behave differently, this notebook reports both:

- all gaps
- gaps starting after \(p_n \ge 100\)

In [ ]:
gap_df = pd.DataFrame({
    "n": np.arange(1, len(gaps) + 1),
    "p_n": gap_start_primes,
    "p_next": gap_end_primes,
    "gap": gaps,
    "log_p_n": log_baseline,
})

gap_df["absolute_error_vs_log"] = np.abs(gap_df["gap"] - gap_df["log_p_n"])
gap_df["relative_drift_vs_log"] = gap_df["absolute_error_vs_log"] / gap_df["log_p_n"]

gap_df.head()

In [ ]:
analysis_df = gap_df[gap_df["p_n"] >= 100].copy()

mean_gap = float(gap_df["gap"].mean())
mean_log = float(gap_df["log_p_n"].mean())
median_gap = float(gap_df["gap"].median())
max_gap = int(gap_df["gap"].max())

mean_gap_after_100 = float(analysis_df["gap"].mean())
mean_log_after_100 = float(analysis_df["log_p_n"].mean())
median_gap_after_100 = float(analysis_df["gap"].median())
max_gap_after_100 = int(analysis_df["gap"].max())

mean_relative_drift = float(analysis_df["relative_drift_vs_log"].mean())
median_relative_drift = float(analysis_df["relative_drift_vs_log"].median())

# Bounded score: 1/(1+mean drift), so it stays in (0,1].
cgcs_gap_logscale = float(1.0 / (1.0 + mean_relative_drift))

measurement = {
    "mean_gap_all": mean_gap,
    "mean_log_all": mean_log,
    "median_gap_all": median_gap,
    "max_gap_all": max_gap,
    "mean_gap_after_100": mean_gap_after_100,
    "mean_log_after_100": mean_log_after_100,
    "median_gap_after_100": median_gap_after_100,
    "max_gap_after_100": max_gap_after_100,
    "mean_relative_drift_after_100": mean_relative_drift,
    "median_relative_drift_after_100": median_relative_drift,
    "cgcs_gap_logscale": cgcs_gap_logscale,
}

cgcs = {
    "score": cgcs_gap_logscale,
    "definition": "CGCS_gap_logscale = 1 / (1 + mean(|g_n - log(p_n)|/log(p_n))) for p_n >= 100",
    "interpretation": "Closer to 1 indicates lower average drift from logarithmic gap scale.",
}

measurement

## 5. Scale windows

Compare observed average gaps to \(\log(x)\) across increasing scale windows.

This helps separate local fluctuation from scale trend.

In [ ]:
# Bin by p_n scale.
bins = np.array([2, 10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000, 300_000, 1_000_000])
labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

gap_df["scale_bin"] = pd.cut(gap_df["p_n"], bins=bins, labels=labels, include_lowest=True)

scale_df = (
    gap_df.dropna(subset=["scale_bin"])
    .groupby("scale_bin", observed=True)
    .agg(
        p_min=("p_n", "min"),
        p_max=("p_n", "max"),
        gap_count=("gap", "count"),
        mean_gap=("gap", "mean"),
        median_gap=("gap", "median"),
        max_gap=("gap", "max"),
        mean_log=("log_p_n", "mean"),
        mean_relative_drift=("relative_drift_vs_log", "mean"),
    )
    .reset_index()
)

scale_df["cgcs_bin"] = 1.0 / (1.0 + scale_df["mean_relative_drift"])
scale_df

## 6. Recoverability note

Log scaling partially recovers coarse gap behavior.

It does **not** recover exact gaps.

\[
\mathbb{E}[g_n] \approx \log(p_n)
\]

means average spacing grows roughly like logarithm of scale.

It does not mean

\[
g_n = \log(p_n)
\]

for each prime.

In [ ]:
# Coarse recoverability proxy:
# How well does bin-level mean log scale recover bin-level mean gap?

scale_df["bin_abs_error"] = np.abs(scale_df["mean_gap"] - scale_df["mean_log"])
scale_df["bin_relative_error"] = scale_df["bin_abs_error"] / scale_df["mean_log"]
recoverability_score = float(1.0 / (1.0 + scale_df["bin_relative_error"].mean()))

recoverability = {
    "recoverability_score_bin_mean": recoverability_score,
    "definition": "1 / (1 + mean bin-level relative error between mean gap and mean log baseline)",
    "note": "log scaling recovers coarse bin-level gap scale, not exact gaps",
}

recoverability

## 7. Visualization

Figures exported:

1. gap histogram  
2. gaps versus \(p_n\) with \(\log(p_n)\) baseline  
3. mean gap versus mean \(\log(p_n)\) by scale bin

In [ ]:
# Figure 1: gap histogram.

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(gap_df["gap"], bins=60)
ax.set_title("Prime gap histogram")
ax.set_xlabel("gap")
ax.set_ylabel("frequency")
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_histogram.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

In [ ]:
# Figure 2: gaps across scale with logarithmic baseline.

sample_df = gap_df.iloc[::max(1, len(gap_df)//5000)].copy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sample_df["p_n"], sample_df["gap"], s=4, alpha=0.5, label="observed gaps")
ax.plot(sample_df["p_n"], sample_df["log_p_n"], linewidth=2, label="log(p_n) baseline")
ax.set_xscale("log")
ax.set_title("Prime gaps across scale")
ax.set_xlabel("p_n")
ax.set_ylabel("gap")
ax.legend()
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_gaps_vs_log_baseline.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

In [ ]:
# Figure 3: bin-level mean gap vs mean log baseline.

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scale_df["p_max"], scale_df["mean_gap"], marker="o", label="mean gap")
ax.plot(scale_df["p_max"], scale_df["mean_log"], marker="o", label="mean log(p_n)")
ax.set_xscale("log")
ax.set_title("Mean gap versus logarithmic scale by bin")
ax.set_xlabel("scale bin upper edge")
ax.set_ylabel("value")
ax.legend()
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_mean_gap_vs_log_by_scale.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 8. Interpretation

1. **What continues?**  
   The ordered prime sequence continues across scale.

2. **What remains under constraint?**  
   Mean gap behavior remains near logarithmic scale.

3. **What drifts?**  
   Individual gaps drift around the logarithmic baseline. Larger gaps are measured as deviations, not treated as failure.

4. **What is recoverable?**  
   Coarse scale behavior is recoverable from \(\log(p_n)\). Exact gaps are not recovered by this baseline alone.

5. **What should not be overclaimed?**  
   This notebook does not predict prime gaps exactly.

In [ ]:
interpretation = f'''
# {NOTEBOOK_TITLE}

## Constraint result

This notebook measured prime gaps

g_n = p_(n+1) - p_n

for primes below {N_MAX:,} and compared gap behavior to the logarithmic baseline log(p_n).

## Continues

The ordered prime sequence continues across scale. Gaps provide local measurements of that continuation.

## Remains under constraint

Mean gap behavior remains comparable to logarithmic scale. For p_n >= 100:

- mean gap = {mean_gap_after_100:.6f}
- mean log(p_n) = {mean_log_after_100:.6f}
- median gap = {median_gap_after_100:.6f}
- max gap = {max_gap_after_100}

## Drift

Drift was measured as |g_n - log(p_n)| / log(p_n).

- mean relative drift after 100 = {mean_relative_drift:.6f}
- median relative drift after 100 = {median_relative_drift:.6f}

## CGCS score

CGCS_gap_logscale = 1 / (1 + mean relative drift)

Measured score:

- CGCS_gap_logscale = {cgcs_gap_logscale:.6f}

## Recoverability

The logarithmic baseline recovers coarse gap scale, especially in aggregate windows. It does not recover exact gaps.

## Caution

This notebook does not solve prime gaps, prove RH, or claim exact predictability. It measures gap structure and drift relative to a classical scale baseline.
'''.strip()

print(interpretation)

## 9. Export data, notes, math, and TeX

This block writes:

- numbered CSV files
- metadata JSON
- interpretation markdown
- design notes markdown
- summary TeX snippet
- standalone math notes TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    **recoverability,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
gaps_path = DATA_DIR / f"{NOTEBOOK_NUM}_prime_gaps.csv"
scale_path = DATA_DIR / f"{NOTEBOOK_NUM}_scale_windows.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
gap_df.to_csv(gaps_path, index=False)
scale_df.to_csv(scale_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "recoverability": recoverability,
    "cgcs": cgcs,
    "figures": [str(fig1_path), str(fig2_path), str(fig3_path)],
    "data": {
        "summary": str(summary_path),
        "prime_gaps": str(gaps_path),
        "scale_windows": str(scale_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + "\n", encoding="utf-8")

design_notes = f'''
# Design Notes — {NOTEBOOK_TITLE}

## Notebook role

Notebook 02 follows Notebook 01 by moving from residue constraints to ordered sequence structure.

Notebook 01 measured which residue classes remain under constraint.
Notebook 02 measures how prime order continues through gaps.

## Constraint

Prime gaps are compared to the logarithmic scale baseline:

g_n = p_(n+1) - p_n
E[g_n] approximately log(p_n)

## Measurement

The notebook computes:

1. all prime gaps below N_MAX
2. gap statistics
3. relative drift from log(p_n)
4. scale-window averages
5. coarse recoverability from bin-level logarithmic scale

## CGCS score

CGCS_gap_logscale = 1 / (1 + mean(|g_n - log(p_n)| / log(p_n)))

computed for p_n >= 100.

## Drift

drift_n = |g_n - log(p_n)| / log(p_n)

This makes drift continuous rather than binary.

## Recoverability

Log scaling recovers coarse gap scale, not exact gap identity.

## Handoff

Notebook 03 should measure density versus x/log(x), connecting gap scale to prime-counting scale.
'''.strip()

design_path.write_text(design_notes + "\n", encoding="utf-8")

summary_tex = rf'''
\section*{{{NOTEBOOK_TITLE}}}

This notebook studies prime gaps
\[
g_n = p_{{n+1}} - p_n
\]
and compares them to the logarithmic scale baseline
\[
\mathbb{{E}}[g_n] \approx \log(p_n).
\]

For primes below {N_MAX:,}, using gaps with $p_n \ge 100$:
\begin{{itemize}}
  \item mean gap $= {mean_gap_after_100:.6f}$
  \item mean $\log(p_n) = {mean_log_after_100:.6f}$
  \item mean relative drift $= {mean_relative_drift:.6f}$
  \item $CGCS_{{gap}} = {cgcs_gap_logscale:.6f}$
\end{{itemize}}

Logarithmic scaling recovers coarse gap scale, not exact gaps.
'''.strip()

summary_tex_path.write_text(summary_tex + "\n", encoding="utf-8")

math_tex = rf"""
\documentclass{{article}}
\usepackage{{amsmath}}
\usepackage{{amssymb}}
\usepackage[margin=1in]{{geometry}}

\begin{{document}}

\section*{{Math Notes: Prime Gaps}}

\subsection*{{Prime gap definition}}

Let $p_n$ denote the $n$-th prime. Define
\[
g_n = p_{{n+1}} - p_n.
\]

\subsection*{{Logarithmic scale baseline}}

The coarse expected gap scale near $p_n$ is
\[
\mathbb{{E}}[g_n] \approx \log(p_n).
\]

\subsection*{{Drift}}

This notebook measures relative drift by
\[
drift_n =
\frac{{|g_n - \log(p_n)|}}{{\log(p_n)}}.
\]

\subsection*{{CGCS score}}

For gaps with $p_n \ge 100$:
\[
CGCS_{{gap}} =
\frac{{1}}{{1 + \frac{{1}}{{N}}\sum_n drift_n}}.
\]

Measured value:
\[
CGCS_{{gap}} = {cgcs_gap_logscale:.6f}.
\]

\subsection*{{Recoverability}}

The logarithmic baseline partially recovers coarse gap scale:
\[
\overline{{g}}_{{\mathrm{{bin}}}} \approx \overline{{\log(p_n)}}_{{\mathrm{{bin}}}}.
\]

It does not recover exact gaps:
\[
g_n \ne \log(p_n)
\]
in general.

\end{{document}}
""".strip()

math_tex_path.write_text(math_tex + "\n", encoding="utf-8")

summary_path, gaps_path, scale_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 10. Export zip

This follows the pi-stage-lab style: root-level export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 11. Next notebook handoff

Next notebook:

```text
03_density_vs_log.ipynb
```

Purpose:

> measure prime density versus \(x/\log(x)\), connecting gap scale to prime-counting scale.

In [ ]:
next_step = "Notebook 03: density versus x/log(x)."
print(next_step)